# 04 — Build Corrected 2020 + 2025 LULC Dataset (Kaggle-only)

This notebook rebuilds the corrected reference-label dataset **directly inside Kaggle**.

### Important
- **No Google Drive exports.**
- The original Kaggle dataset in `/kaggle/input` is never modified.
- 2020 labels are generated from **ESA WorldCover 2020**.
- 2025 labels are generated from **Dynamic World V1 (2025)**.
- Labels are downloaded directly from Earth Engine into `/kaggle/working`.
- The export grid is taken from each existing Sentinel-2 tile so labels match the image grid exactly.
- Final labels:
  - `0 = Water`
  - `1 = Vegetation`
  - `2 = Built-up`
  - `3 = Bare Land`
  - `255 = NoData / Ignore`

The resulting dataset is prepared for the later semantic-transition baseline notebook.


## 01. Imports and configuration


In [18]:
import os
import glob
import shutil
import zipfile
import requests
import time
import json
import warnings
from pathlib import Path

import numpy as np
import rasterio
import ee

warnings.filterwarnings("ignore")

PROJECT_ID = "sentinel-lulc-project-510412"

# Original Kaggle dataset candidates used by the project.
INPUT_CANDIDATES = [
    "/kaggle/input/datasets/mssamiksha30/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset_v2",
    "/kaggle/input/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset_v2",
]

WORK_BASE = "/kaggle/working/sentinel2_lulc_change_dataset_v2_corrected"

print("Configuration loaded.")


Configuration loaded.


## 02. Locate the original Kaggle dataset


In [19]:
INPUT_BASE = None

for candidate in INPUT_CANDIDATES:
    if os.path.isdir(candidate):
        INPUT_BASE = candidate
        break

if INPUT_BASE is None:
    print("The two known paths were not found.")
    print("Searching /kaggle/input for sentinel2_lulc_change_dataset_v2 ...")

    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "sentinel2_lulc_change_dataset_v2":
            if os.path.isdir(os.path.join(root, "2020")) and os.path.isdir(os.path.join(root, "2025")):
                INPUT_BASE = root
                break

if INPUT_BASE is None:
    raise FileNotFoundError(
        "Could not find the original sentinel2_lulc_change_dataset_v2 dataset under /kaggle/input."
    )

print("Original dataset:")
print(INPUT_BASE)


Original dataset:
/kaggle/input/datasets/mssamiksha30/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset_v2


## 03. Create a clean working copy

This copies the existing Sentinel-2 images and metadata only. The original Kaggle input remains untouched.


In [20]:
if os.path.exists(WORK_BASE):
    shutil.rmtree(WORK_BASE)

os.makedirs(WORK_BASE, exist_ok=True)

for folder in ["2020", "2025", "metadata", "patches"]:
    src = os.path.join(INPUT_BASE, folder)
    dst = os.path.join(WORK_BASE, folder)

    if os.path.isdir(src):
        shutil.copytree(src, dst)

print("Working copy created:")
print(WORK_BASE)
print("Original dataset was not modified.")


Working copy created:
/kaggle/working/sentinel2_lulc_change_dataset_v2_corrected
Original dataset was not modified.


## 04. Verify the existing Sentinel-2 tiles


In [21]:
IMG2020_DIR = os.path.join(WORK_BASE, "2020")
IMG2025_DIR = os.path.join(WORK_BASE, "2025")

img2020 = sorted(glob.glob(os.path.join(IMG2020_DIR, "*.tif")))
img2025 = sorted(glob.glob(os.path.join(IMG2025_DIR, "*.tif")))

print("2020 images:", len(img2020))
print("2025 images:", len(img2025))

assert len(img2020) == 63, f"Expected 63 2020 tiles, found {len(img2020)}"
assert len(img2025) == 63, f"Expected 63 2025 tiles, found {len(img2025)}"

names2020 = [os.path.basename(p) for p in img2020]
names2025 = [os.path.basename(p) for p in img2025]

assert names2020 == names2025, "2020 and 2025 tile filenames do not match."

with rasterio.open(img2020[0]) as src:
    print("Example 2020 shape:", (src.height, src.width))
    print("Example 2020 CRS:", src.crs)
    print("Example 2020 channels:", src.count)

with rasterio.open(img2025[0]) as src:
    print("Example 2025 shape:", (src.height, src.width))
    print("Example 2025 CRS:", src.crs)
    print("Example 2025 channels:", src.count)

assert rasterio.open(img2020[0]).count == 9
assert rasterio.open(img2025[0]).count == 9

print("✓ 63 paired Sentinel-2 tiles found.")
print("✓ Each image has 9 channels.")


2020 images: 63
2025 images: 63
Example 2020 shape: (557, 558)
Example 2020 CRS: EPSG:4326
Example 2020 channels: 9
Example 2025 shape: (557, 558)
Example 2025 CRS: EPSG:4326
Example 2025 channels: 9
✓ 63 paired Sentinel-2 tiles found.
✓ Each image has 9 channels.


## 05. Initialize Google Earth Engine


In [22]:
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized.")
print("Project:", PROJECT_ID)


Earth Engine initialized.
Project: sentinel-lulc-project-510412


## 06. Define the Hyderabad study region


In [23]:
HYDERABAD = ee.Geometry.Rectangle(
    [78.20, 17.20, 78.70, 17.60],
    geodesic=False
)

print("Hyderabad study region defined.")


Hyderabad study region defined.


## 07. Helper: download one Earth Engine image directly into Kaggle

This is the replacement for the earlier Google Drive export approach.

For every tile, the helper reads the **exact CRS, transform, width and height from the existing Sentinel-2 TIFF**, then asks Earth Engine for a GeoTIFF on that same grid.


In [33]:
# ============================================================
# FINAL FIXED EARTH ENGINE → KAGGLE TIFF DOWNLOADER
# ============================================================

import os
import time
import requests
import rasterio


def download_ee_image_to_tif(
    ee_image,
    output_path,
    reference_path
):
    """
    Download an Earth Engine image directly to Kaggle
    using the EXACT grid of the Sentinel-2 reference TIFF.

    The output uses:
        - same width
        - same height
        - same CRS
        - same affine transform

    No Google Drive is used.
    """

    # --------------------------------------------------------
    # Read exact Sentinel-2 raster grid
    # --------------------------------------------------------
    with rasterio.open(reference_path) as ref:
        width = ref.width
        height = ref.height
        crs = ref.crs
        transform = ref.transform

    # --------------------------------------------------------
    # Convert rasterio Affine transform to Earth Engine format
    # --------------------------------------------------------
    ee_transform = [
        transform.a,
        transform.b,
        transform.c,
        transform.d,
        transform.e,
        transform.f
    ]

    # --------------------------------------------------------
    # Earth Engine download parameters
    #
    # IMPORTANT:
    # Do NOT include "region" here.
    # We are specifying the exact grid using
    # dimensions + crs + crs_transform.
    # --------------------------------------------------------
    params = {
        "name": os.path.splitext(
            os.path.basename(output_path)
        )[0],

        "dimensions": [width, height],

        "crs": crs.to_string(),

        "crs_transform": ee_transform,

        "format": "GEO_TIFF"
    }

    # --------------------------------------------------------
    # Try download up to 3 times
    # --------------------------------------------------------
    last_error = None

    for attempt in range(1, 4):

        try:

            print(
                f"Downloading "
                f"{os.path.basename(output_path)} "
                f"(attempt {attempt}/3)..."
            )

            # Get temporary download URL
            url = ee_image.getDownloadURL(params)

            # Download directly into Kaggle
            response = requests.get(
                url,
                timeout=300
            )

            response.raise_for_status()

            # Save TIFF
            with open(output_path, "wb") as f:
                f.write(response.content)

            # ------------------------------------------------
            # Verify file
            # ------------------------------------------------
            if not os.path.exists(output_path):
                raise RuntimeError(
                    "Output file was not created."
                )

            if os.path.getsize(output_path) == 0:
                raise RuntimeError(
                    "Downloaded file is empty."
                )

            print(
                f"✓ Saved: "
                f"{os.path.basename(output_path)}"
            )

            return

        except Exception as e:

            last_error = e

            print(
                f"✗ Attempt {attempt} failed: {e}"
            )

            if attempt < 3:
                time.sleep(5)

    raise RuntimeError(
        f"Failed after 3 attempts: {last_error}"
    )

## 08. Prepare corrected 2020 ESA WorldCover labels


In [34]:
worldcover_2020 = ee.Image("ESA/WorldCover/v100/2020").select("Map")

# Project encoding:
# 0 = Water
# 1 = Vegetation
# 2 = Built-up
# 3 = Bare Land
# 255 = Ignore / NoData

label_2020 = worldcover_2020.remap(
    [
        80, 90,                       # Water
        10, 20, 30, 40, 95, 100,     # Vegetation
        50,                           # Built-up
        60                            # Bare Land
    ],
    [
        0,                            # Water
        0,
        1, 1, 1, 1, 1, 1,             # Vegetation
        2,                            # Built-up
        3                             # Bare Land
    ],
    255
).rename("label").toUint8()

print("✓ 2020 WorldCover label image prepared.")


✓ 2020 WorldCover label image prepared.


## 09. Download corrected 2020 labels directly into Kaggle

**This cell may take some time because it downloads 63 tiles one by one.**
There is no Google Drive step.


In [37]:
LABEL2020_DIR = os.path.join(WORK_BASE, "labels_2020_corrected")

if os.path.exists(LABEL2020_DIR):
    shutil.rmtree(LABEL2020_DIR)
os.makedirs(LABEL2020_DIR, exist_ok=True)

for i, image_path in enumerate(img2020, 1):
    fname = os.path.basename(image_path)
    out_path = os.path.join(LABEL2020_DIR, fname)

    with rasterio.open(image_path) as src:
        bounds = src.bounds
        region = [
            [bounds.left, bounds.bottom],
            [bounds.right, bounds.bottom],
            [bounds.right, bounds.top],
            [bounds.left, bounds.top],
            [bounds.left, bounds.bottom],
        ]

    tile_label = label_2020.clip(
        ee.Geometry.Polygon(region, geodesic=False)
    )

    if os.path.exists(out_path):
        print(f"[{i:02d}/63] already exists: {fname}")
        continue

    download_ee_image_to_tif(
        tile_label,
        out_path,
        image_path,
        
    )

    print(f"[{i:02d}/63] downloaded: {fname}")

print()
print("✓ 2020 label download complete.")
print("Files:", len(glob.glob(os.path.join(LABEL2020_DIR, "*.tif"))))


✓ Saved: tile_000.tif
[01/63] downloaded: tile_000.tif
✓ Saved: tile_001.tif
[02/63] downloaded: tile_001.tif
✓ Saved: tile_002.tif
[03/63] downloaded: tile_002.tif
✓ Saved: tile_003.tif
[04/63] downloaded: tile_003.tif
✓ Saved: tile_004.tif
[05/63] downloaded: tile_004.tif
✓ Saved: tile_005.tif
[06/63] downloaded: tile_005.tif
✓ Saved: tile_006.tif
[07/63] downloaded: tile_006.tif
✓ Saved: tile_007.tif
[08/63] downloaded: tile_007.tif
✓ Saved: tile_008.tif
[09/63] downloaded: tile_008.tif
✓ Saved: tile_009.tif
[10/63] downloaded: tile_009.tif
✓ Saved: tile_010.tif
[11/63] downloaded: tile_010.tif
✓ Saved: tile_011.tif
[12/63] downloaded: tile_011.tif
✓ Saved: tile_012.tif
[13/63] downloaded: tile_012.tif
✓ Saved: tile_013.tif
[14/63] downloaded: tile_013.tif
✓ Saved: tile_014.tif
[15/63] downloaded: tile_014.tif
✓ Saved: tile_015.tif
[16/63] downloaded: tile_015.tif
✓ Saved: tile_016.tif
[17/63] downloaded: tile_016.tif
✓ Saved: tile_017.tif
[18/63] downloaded: tile_017.tif
✓ Saved: t

## 10. Validate corrected 2020 labels


In [38]:
label2020 = sorted(glob.glob(os.path.join(LABEL2020_DIR, "*.tif")))

assert len(label2020) == 63

global_counts_2020 = {0: 0, 1: 0, 2: 0, 3: 0, 255: 0}

for img_path, lab_path in zip(img2020, label2020):
    assert os.path.basename(img_path) == os.path.basename(lab_path)

    with rasterio.open(img_path) as img, rasterio.open(lab_path) as lab:
        assert img.width == lab.width
        assert img.height == lab.height
        assert img.crs == lab.crs
        assert img.transform == lab.transform
        arr = lab.read(1)

    unique, counts = np.unique(arr, return_counts=True)

    for u, c in zip(unique, counts):
        assert int(u) in global_counts_2020, f"Unexpected 2020 label value: {u}"
        global_counts_2020[int(u)] += int(c)

print("2020 global label counts:")
for k, v in global_counts_2020.items():
    print(f"{k:3d}: {v:,}")

assert global_counts_2020[0] > 0, "No valid Water pixels found."
assert global_counts_2020[1] > 0, "No Vegetation pixels found."
assert global_counts_2020[2] > 0, "No Built-up pixels found."
assert global_counts_2020[3] > 0, "No Bare Land pixels found."

print()
print("✓ 2020 labels are valid.")
print("✓ 2020 labels match the Sentinel-2 grids exactly.")


2020 global label counts:
  0: 192,323
  1: 12,027,707
  2: 5,972,277
  3: 1,392,947
255: 0

✓ 2020 labels are valid.
✓ 2020 labels match the Sentinel-2 grids exactly.


## 11. Prepare 2025 Dynamic World reference labels

Dynamic World V1 is used for 2025 because the original dataset preparation did not contain a 2025 reference-label raster.

The Dynamic World probability bands are median-composited over 2025, then the highest-probability class is selected.

Project mapping:
- water → 0
- trees/grass/flooded vegetation/crops/shrub & scrub → 1
- built → 2
- bare → 3
- snow & ice → 255


In [39]:
dw_collection = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(HYDERABAD)
    .filterDate("2025-01-01", "2026-01-01")
)

dw_count = dw_collection.size().getInfo()
print("Dynamic World observations in 2025:", dw_count)

assert dw_count > 0

probability_bands = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
]

dw_probability = dw_collection.select(probability_bands).median()

dw_class = (
    dw_probability
    .toArray()
    .arrayArgmax()
    .arrayGet([0])
)

# Project encoding.
label_2025 = ee.Image.constant(255).rename("label")

label_2025 = label_2025.where(dw_class.eq(0), 0)  # water

for dw_cls in [1, 2, 3, 4, 5]:
    label_2025 = label_2025.where(dw_class.eq(dw_cls), 1)

label_2025 = label_2025.where(dw_class.eq(6), 2)  # built
label_2025 = label_2025.where(dw_class.eq(7), 3)  # bare

# 8 = snow & ice remains 255.
label_2025 = label_2025.toUint8()

print("✓ 2025 Dynamic World project labels prepared.")


Dynamic World observations in 2025: 147
✓ 2025 Dynamic World project labels prepared.


## 12. Download corrected 2025 labels directly into Kaggle

Again, this uses **direct Earth Engine → Kaggle download**. No Drive.


In [41]:
LABEL2025_DIR = os.path.join(WORK_BASE, "labels_2025_aligned")

if os.path.exists(LABEL2025_DIR):
    shutil.rmtree(LABEL2025_DIR)
os.makedirs(LABEL2025_DIR, exist_ok=True)

for i, image_path in enumerate(img2025, 1):
    fname = os.path.basename(image_path)
    out_path = os.path.join(LABEL2025_DIR, fname)

    with rasterio.open(image_path) as src:
        bounds = src.bounds
        region = [
            [bounds.left, bounds.bottom],
            [bounds.right, bounds.bottom],
            [bounds.right, bounds.top],
            [bounds.left, bounds.top],
            [bounds.left, bounds.bottom],
        ]

    tile_label = label_2025.clip(
        ee.Geometry.Polygon(region, geodesic=False)
    )

    if os.path.exists(out_path):
        print(f"[{i:02d}/63] already exists: {fname}")
        continue

    download_ee_image_to_tif(
        tile_label,
        out_path,
        image_path
    )

    print(f"[{i:02d}/63] downloaded: {fname}")

print()
print("✓ 2025 label download complete.")
print("Files:", len(glob.glob(os.path.join(LABEL2025_DIR, "*.tif"))))


✓ Saved: tile_000.tif
[01/63] downloaded: tile_000.tif
✓ Saved: tile_001.tif
[02/63] downloaded: tile_001.tif
✓ Saved: tile_002.tif
[03/63] downloaded: tile_002.tif
✓ Saved: tile_003.tif
[04/63] downloaded: tile_003.tif
✓ Saved: tile_004.tif
[05/63] downloaded: tile_004.tif
✓ Saved: tile_005.tif
[06/63] downloaded: tile_005.tif
✓ Saved: tile_006.tif
[07/63] downloaded: tile_006.tif
✓ Saved: tile_007.tif
[08/63] downloaded: tile_007.tif
✓ Saved: tile_008.tif
[09/63] downloaded: tile_008.tif
✓ Saved: tile_009.tif
[10/63] downloaded: tile_009.tif
✓ Saved: tile_010.tif
[11/63] downloaded: tile_010.tif
✓ Saved: tile_011.tif
[12/63] downloaded: tile_011.tif
✓ Saved: tile_012.tif
[13/63] downloaded: tile_012.tif
✓ Saved: tile_013.tif
[14/63] downloaded: tile_013.tif
✓ Saved: tile_014.tif
[15/63] downloaded: tile_014.tif
✓ Saved: tile_015.tif
[16/63] downloaded: tile_015.tif
✓ Saved: tile_016.tif
[17/63] downloaded: tile_016.tif
✓ Saved: tile_017.tif
[18/63] downloaded: tile_017.tif
✓ Saved: t

## 13. Validate corrected 2025 labels


In [42]:
label2025 = sorted(glob.glob(os.path.join(LABEL2025_DIR, "*.tif")))

assert len(label2025) == 63

global_counts_2025 = {0: 0, 1: 0, 2: 0, 3: 0, 255: 0}

for img_path, lab_path in zip(img2025, label2025):
    assert os.path.basename(img_path) == os.path.basename(lab_path)

    with rasterio.open(img_path) as img, rasterio.open(lab_path) as lab:
        assert img.width == lab.width
        assert img.height == lab.height
        assert img.crs == lab.crs
        assert img.transform == lab.transform
        arr = lab.read(1)

    unique, counts = np.unique(arr, return_counts=True)

    for u, c in zip(unique, counts):
        assert int(u) in global_counts_2025, f"Unexpected 2025 label value: {u}"
        global_counts_2025[int(u)] += int(c)

print("2025 global label counts:")
for k, v in global_counts_2025.items():
    print(f"{k:3d}: {v:,}")

assert global_counts_2025[0] > 0
assert global_counts_2025[1] > 0
assert global_counts_2025[2] > 0
assert global_counts_2025[3] > 0

print()
print("✓ 2025 labels are valid.")
print("✓ 2025 labels match the Sentinel-2 grids exactly.")


2025 global label counts:
  0: 834,452
  1: 7,242,532
  2: 11,408,758
  3: 99,349
255: 163

✓ 2025 labels are valid.
✓ 2025 labels match the Sentinel-2 grids exactly.


## 14. Final image + label validation

This is the important checkpoint before building the final dataset.


In [43]:
assert len(img2020) == 63
assert len(img2025) == 63
assert len(label2020) == 63
assert len(label2025) == 63

for img_path, lab_path in zip(img2020, label2020):
    with rasterio.open(img_path) as img, rasterio.open(lab_path) as lab:
        assert img.width == lab.width
        assert img.height == lab.height
        assert img.crs == lab.crs
        assert img.transform == lab.transform
        assert img.count == 9
        assert lab.count == 1

for img_path, lab_path in zip(img2025, label2025):
    with rasterio.open(img_path) as img, rasterio.open(lab_path) as lab:
        assert img.width == lab.width
        assert img.height == lab.height
        assert img.crs == lab.crs
        assert img.transform == lab.transform
        assert img.count == 9
        assert lab.count == 1

print("=" * 70)
print("FINAL LABEL + IMAGE VALIDATION PASSED")
print("=" * 70)
print("2020 images :", len(img2020))
print("2025 images :", len(img2025))
print("2020 labels :", len(label2020))
print("2025 labels :", len(label2025))
print()
print("✓ Filenames match")
print("✓ Shape matches")
print("✓ CRS matches")
print("✓ Transform matches")
print("✓ 9 image channels")
print("✓ Valid label classes 0/1/2/3/255")


FINAL LABEL + IMAGE VALIDATION PASSED
2020 images : 63
2025 images : 63
2020 labels : 63
2025 labels : 63

✓ Filenames match
✓ Shape matches
✓ CRS matches
✓ Transform matches
✓ 9 image channels
✓ Valid label classes 0/1/2/3/255


## 15. Assemble the final dataset

Run this only after the validation cell above passes.


In [44]:
FINAL_DIR = "/kaggle/working/sentinel2_lulc_change_dataset_FINAL"

if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)

os.makedirs(FINAL_DIR, exist_ok=True)

# Copy Sentinel-2 images.
shutil.copytree(
    IMG2020_DIR,
    os.path.join(FINAL_DIR, "2020")
)

shutil.copytree(
    IMG2025_DIR,
    os.path.join(FINAL_DIR, "2025")
)

# Copy corrected labels using the final folder names.
shutil.copytree(
    LABEL2020_DIR,
    os.path.join(FINAL_DIR, "labels_2020")
)

shutil.copytree(
    LABEL2025_DIR,
    os.path.join(FINAL_DIR, "labels_2025")
)

# Preserve existing metadata/splits.
metadata_src = os.path.join(WORK_BASE, "metadata")
if os.path.isdir(metadata_src):
    shutil.copytree(
        metadata_src,
        os.path.join(FINAL_DIR, "metadata")
    )

print("=" * 70)
print("FINAL DATASET ASSEMBLED")
print("=" * 70)
print(FINAL_DIR)


FINAL DATASET ASSEMBLED
/kaggle/working/sentinel2_lulc_change_dataset_FINAL


## 16. Final dataset structure check


In [45]:
for folder in ["2020", "2025", "labels_2020", "labels_2025"]:
    path = os.path.join(FINAL_DIR, folder)
    files = sorted(glob.glob(os.path.join(path, "*.tif")))
    print(f"{folder:12s}: {len(files)} TIFF files")
    assert len(files) == 63

print()
print("Final dataset:")
for root, dirs, files in os.walk(FINAL_DIR):
    level = root.replace(FINAL_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in sorted(files)[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ...")

print()
print("✓ DATASET BUILD COMPLETE")
print("✓ No Google Drive step was used.")
print("✓ Ready for transition-label generation / baseline preparation.")


2020        : 63 TIFF files
2025        : 63 TIFF files
labels_2020 : 63 TIFF files
labels_2025 : 63 TIFF files

Final dataset:
sentinel2_lulc_change_dataset_FINAL/
  2025/
    tile_000.tif
    tile_001.tif
    tile_002.tif
    tile_003.tif
    tile_004.tif
    ...
  labels_2020/
    tile_000.tif
    tile_001.tif
    tile_002.tif
    tile_003.tif
    tile_004.tif
    ...
  metadata/
    patch_metadata.csv
  2020/
    tile_000.tif
    tile_001.tif
    tile_002.tif
    tile_003.tif
    tile_004.tif
    ...
  labels_2025/
    tile_000.tif
    tile_001.tif
    tile_002.tif
    tile_003.tif
    tile_004.tif
    ...

✓ DATASET BUILD COMPLETE
✓ No Google Drive step was used.
✓ Ready for transition-label generation / baseline preparation.


In [46]:
import os
import glob
import rasterio
from collections import Counter

base = "/kaggle/working/sentinel2_lulc_change_dataset_v2_corrected"

folders = {
    "2020 images": os.path.join(base, "2020"),
    "2025 images": os.path.join(base, "2025"),
    "2020 labels": os.path.join(base, "labels_2020_aligned"),
    "2025 labels": os.path.join(base, "labels_2025_aligned"),
}

print("=" * 70)
print("FINAL DATASET CHECK")
print("=" * 70)

for name, folder in folders.items():
    files = sorted(glob.glob(os.path.join(folder, "*.tif")))
    print(f"{name:20s}: {len(files)} TIFFs")

print("\nChecking image-label alignment...")

img20 = sorted(glob.glob(os.path.join(folders["2020 images"], "*.tif")))
img25 = sorted(glob.glob(os.path.join(folders["2025 images"], "*.tif")))
lab20 = sorted(glob.glob(os.path.join(folders["2020 labels"], "*.tif")))
lab25 = sorted(glob.glob(os.path.join(folders["2025 labels"], "*.tif")))

assert len(img20) == len(img25) == len(lab20) == len(lab25) == 63

for i in range(63):

    with rasterio.open(img20[i]) as a:
        img20_shape = (a.height, a.width)
        img20_crs = a.crs
        img20_transform = a.transform

    with rasterio.open(lab20[i]) as a:
        lab20_shape = (a.height, a.width)
        lab20_crs = a.crs
        lab20_transform = a.transform

    with rasterio.open(img25[i]) as a:
        img25_shape = (a.height, a.width)
        img25_crs = a.crs
        img25_transform = a.transform

    with rasterio.open(lab25[i]) as a:
        lab25_shape = (a.height, a.width)
        lab25_crs = a.crs
        lab25_transform = a.transform

    assert img20_shape == lab20_shape
    assert img25_shape == lab25_shape

    assert img20_crs == lab20_crs
    assert img25_crs == lab25_crs

    assert img20_transform == lab20_transform
    assert img25_transform == lab25_transform

print("✓ 2020 image-label grids match")
print("✓ 2025 image-label grids match")

print("\nChecking label values...")

allowed = {0, 1, 2, 3, 255}

for label_folder in [
    folders["2020 labels"],
    folders["2025 labels"]
]:

    for path in sorted(glob.glob(os.path.join(label_folder, "*.tif"))):

        with rasterio.open(path) as src:
            values = set(src.read(1).flatten().tolist())

        assert values.issubset(allowed), (
            f"Invalid values in {os.path.basename(path)}: "
            f"{values - allowed}"
        )

print("✓ All labels contain only:")
print("  0   = Water")
print("  1   = Vegetation")
print("  2   = Built-up")
print("  3   = Bare Land")
print("  255 = NoData / Ignore")

print("\n" + "=" * 70)
print("✓ FINAL DATASET VALIDATION PASSED")
print("=" * 70)

FINAL DATASET CHECK
2020 images         : 63 TIFFs
2025 images         : 63 TIFFs
2020 labels         : 0 TIFFs
2025 labels         : 63 TIFFs

Checking image-label alignment...


AssertionError: 

In [48]:
import os
import glob
import rasterio

base = "/kaggle/working/sentinel2_lulc_change_dataset_v2_corrected"

img20 = sorted(glob.glob(os.path.join(base, "2020", "*.tif")))
lab20 = sorted(glob.glob(os.path.join(base, "labels_2020_corrected", "*.tif")))

print("2020 images :", len(img20))
print("2020 labels :", len(lab20))

assert len(img20) == 63
assert len(lab20) == 63

mismatches = []

for img_path, lab_path in zip(img20, lab20):

    with rasterio.open(img_path) as img, rasterio.open(lab_path) as lab:

        checks = {
            "shape": (img.width == lab.width and img.height == lab.height),
            "crs": (img.crs == lab.crs),
            "transform": (img.transform == lab.transform),
        }

        if not all(checks.values()):
            mismatches.append({
                "file": os.path.basename(img_path),
                **checks,
                "image_shape": (img.width, img.height),
                "label_shape": (lab.width, lab.height)
            })

print("\nMismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatches:")
    for x in mismatches[:10]:
        print(x)
else:
    print("\n✓ ALL 63 2020 LABELS MATCH THE 2020 SENTINEL-2 GRID")

2020 images : 63
2020 labels : 63

Mismatches: 0

✓ ALL 63 2020 LABELS MATCH THE 2020 SENTINEL-2 GRID


In [49]:
import os
import glob
import rasterio

base = "/kaggle/working/sentinel2_lulc_change_dataset_v2_corrected"

img25 = sorted(glob.glob(os.path.join(base, "2025", "*.tif")))
lab25 = sorted(glob.glob(os.path.join(base, "labels_2025_aligned", "*.tif")))

print("2025 images :", len(img25))
print("2025 labels :", len(lab25))

assert len(img25) == 63
assert len(lab25) == 63

mismatches = []

for img_path, lab_path in zip(img25, lab25):

    with rasterio.open(img_path) as img, rasterio.open(lab_path) as lab:

        checks = {
            "shape": (img.width == lab.width and img.height == lab.height),
            "crs": (img.crs == lab.crs),
            "transform": (img.transform == lab.transform),
        }

        if not all(checks.values()):
            mismatches.append({
                "file": os.path.basename(img_path),
                **checks,
                "image_shape": (img.width, img.height),
                "label_shape": (lab.width, lab.height)
            })

print("\nMismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatches:")
    for x in mismatches[:10]:
        print(x)
else:
    print("\n✓ ALL 63 2025 LABELS MATCH THE 2025 SENTINEL-2 GRID")

2025 images : 63
2025 labels : 63

Mismatches: 0

✓ ALL 63 2025 LABELS MATCH THE 2025 SENTINEL-2 GRID


In [50]:
import os
import shutil

base = "/kaggle/working/sentinel2_lulc_change_dataset_v2_corrected"
final_dir = "/kaggle/working/sentinel2_lulc_change_dataset_FINAL"

# Remove previous incomplete FINAL folder if it exists
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)

os.makedirs(final_dir, exist_ok=True)

# Copy the validated folders
shutil.copytree(
    os.path.join(base, "2020"),
    os.path.join(final_dir, "2020")
)

shutil.copytree(
    os.path.join(base, "2025"),
    os.path.join(final_dir, "2025")
)

shutil.copytree(
    os.path.join(base, "labels_2020_corrected"),
    os.path.join(final_dir, "labels_2020")
)

shutil.copytree(
    os.path.join(base, "labels_2025_aligned"),
    os.path.join(final_dir, "labels_2025")
)

# Preserve metadata/splits if available
metadata_src = os.path.join(base, "metadata")

if os.path.exists(metadata_src):
    shutil.copytree(
        metadata_src,
        os.path.join(final_dir, "metadata")
    )

print("=" * 70)
print("FINAL DATASET CREATED")
print("=" * 70)
print(final_dir)

FINAL DATASET CREATED
/kaggle/working/sentinel2_lulc_change_dataset_FINAL


In [51]:
import os
import glob

final_dir = "/kaggle/working/sentinel2_lulc_change_dataset_FINAL"

for folder in [
    "2020",
    "2025",
    "labels_2020",
    "labels_2025"
]:
    path = os.path.join(final_dir, folder)
    files = glob.glob(os.path.join(path, "*.tif"))
    print(f"{folder:15s}: {len(files)} TIFFs")

print("\nFinal dataset:")
for root, dirs, files in os.walk(final_dir):
    level = root.replace(final_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

2020           : 63 TIFFs
2025           : 63 TIFFs
labels_2020    : 63 TIFFs
labels_2025    : 63 TIFFs

Final dataset:
sentinel2_lulc_change_dataset_FINAL/
  2025/
  labels_2020/
  metadata/
  2020/
  labels_2025/


In [60]:
import os
import shutil

# Your validated final dataset
source = "/kaggle/working/sentinel2_lulc_change_dataset_FINAL"

# Desired clean name
dataset_name = "sentinel2_lulc_change_dataset_v2"
clean_folder = f"/kaggle/working/{dataset_name}"
zip_path = f"/kaggle/working/{dataset_name}"

# Remove old copy/zip if present
if os.path.exists(clean_folder):
    shutil.rmtree(clean_folder)

if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Copy FINAL dataset and rename it cleanly
shutil.copytree(source, clean_folder)

# Create ZIP
shutil.make_archive(
    zip_path,
    "zip",
    root_dir="/kaggle/working",
    base_dir=dataset_name
)

print("=" * 70)
print("ZIP CREATED")
print("=" * 70)
print(f"Folder : {clean_folder}")
print(f"ZIP    : {zip_path}.zip")
print(f"Size   : {os.path.getsize(zip_path + '.zip') / (1024**3):.2f} GB")

ZIP CREATED
Folder : /kaggle/working/sentinel2_lulc_change_dataset_v2
ZIP    : /kaggle/working/sentinel2_lulc_change_dataset_v2.zip
Size   : 0.93 GB


In [64]:
import zipfile

zip_path = "/kaggle/working/sentinel2_lulc_change_dataset_v2.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

print("ZIP contents:")
print()

for folder in [
    "2020/",
    "2025/",
    "labels_2020/",
    "labels_2025/",
    "metadata/"
]:
    count = sum(
        1 for f in files
        if f.startswith("sentinel2_lulc_change_dataset_v2/" + folder)
        and f.lower().endswith(".tif")
    )
    print(f"{folder:15s} : {count} TIFFs")

print("\n✓ ZIP READY")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/sentinel2_lulc_change_dataset_v3.zip'

In [54]:
import os

final_dir = "/kaggle/working/sentinel2_lulc_change_dataset_FINAL"

print("Folders in FINAL dataset:")
for item in sorted(os.listdir(final_dir)):
    path = os.path.join(final_dir, item)
    if os.path.isdir(path):
        print("📁", item)

Folders in FINAL dataset:
📁 2020
📁 2025
📁 labels_2020
📁 labels_2025
📁 metadata


In [56]:
import pandas as pd
d=pd.read_csv("/kaggle/working/sentinel2_lulc_change_dataset_FINAL/metadata/patch_metadata.csv")
print(d)

                  patch_id parent_tile       split
0     tile_000_patch_00000    tile_000  validation
1     tile_000_patch_00001    tile_000  validation
2     tile_000_patch_00002    tile_000  validation
3     tile_000_patch_00003    tile_000  validation
4     tile_000_patch_00004    tile_000  validation
...                    ...         ...         ...
1003  tile_062_patch_01003    tile_062       train
1004  tile_062_patch_01004    tile_062       train
1005  tile_062_patch_01005    tile_062       train
1006  tile_062_patch_01006    tile_062       train
1007  tile_062_patch_01007    tile_062       train

[1008 rows x 3 columns]


In [57]:
import os

final_dir = "/kaggle/working/sentinel2_lulc_change_dataset_FINAL"

print("=" * 70)
print("FINAL DATASET STRUCTURE")
print("=" * 70)

for root, dirs, files in os.walk(final_dir):
    level = root.replace(final_dir, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for f in files[:10]:
        print(f"{indent}    {f}")

    if len(files) > 10:
        print(f"{indent}    ... ({len(files)} files)")

FINAL DATASET STRUCTURE
sentinel2_lulc_change_dataset_FINAL/
    2025/
        tile_023.tif
        tile_050.tif
        tile_042.tif
        tile_044.tif
        tile_028.tif
        tile_038.tif
        tile_012.tif
        tile_045.tif
        tile_051.tif
        tile_027.tif
        ... (63 files)
    labels_2020/
        tile_023.tif
        tile_050.tif
        tile_042.tif
        tile_044.tif
        tile_028.tif
        tile_038.tif
        tile_012.tif
        tile_045.tif
        tile_051.tif
        tile_027.tif
        ... (63 files)
    metadata/
        patch_metadata.csv
    2020/
        tile_023.tif
        tile_050.tif
        tile_042.tif
        tile_044.tif
        tile_028.tif
        tile_038.tif
        tile_012.tif
        tile_045.tif
        tile_051.tif
        tile_027.tif
        ... (63 files)
    labels_2025/
        tile_023.tif
        tile_050.tif
        tile_042.tif
        tile_044.tif
        tile_028.tif
        tile_038.tif
        tile_012.ti

In [63]:
import os

zip_path = "/kaggle/working/sentinel2_lulc_change_dataset_v2.zip"

print("Exists:", os.path.exists(zip_path))
print("Size:", os.path.getsize(zip_path) / (1024**3), "GB")

Exists: True
Size: 0.9257386038079858 GB


In [69]:
import os

original = "/kaggle/input/datasets/mssamiksha30/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset/patches"
print(original)

print("=" * 80)
print("TOP-LEVEL DATASET BREAKDOWN")
print("=" * 80)

for item in sorted(os.listdir(original)):

    path = os.path.join(original, item)

    if os.path.isdir(path):

        count = 0
        size = 0

        for root, dirs, files in os.walk(path):
            for f in files:
                fp = os.path.join(root, f)
                count += 1
                size += os.path.getsize(fp)

        print(
            f"{item:25s} "
            f"{count:6d} files   "
            f"{size / (1024**3):8.3f} GB"
        )

    else:
        size = os.path.getsize(path)
        print(
            f"{item:25s} "
            f"{1:6d} file     "
            f"{size / (1024**3):8.3f} GB"
        )

/kaggle/input/datasets/mssamiksha30/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset/patches
TOP-LEVEL DATASET BREAKDOWN
2020                        1008 files      0.554 GB
2025                        1008 files      0.554 GB
labels_2020                 1008 files      0.016 GB


In [70]:
import os

original = "/kaggle/input/datasets/mssamiksha30/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset_v2"
print(original)

print("=" * 80)
print("TOP-LEVEL DATASET BREAKDOWN")
print("=" * 80)

for item in sorted(os.listdir(original)):

    path = os.path.join(original, item)

    if os.path.isdir(path):

        count = 0
        size = 0

        for root, dirs, files in os.walk(path):
            for f in files:
                fp = os.path.join(root, f)
                count += 1
                size += os.path.getsize(fp)

        print(
            f"{item:25s} "
            f"{count:6d} files   "
            f"{size / (1024**3):8.3f} GB"
        )

    else:
        size = os.path.getsize(path)
        print(
            f"{item:25s} "
            f"{1:6d} file     "
            f"{size / (1024**3):8.3f} GB"
        )

/kaggle/input/datasets/mssamiksha30/sentinel2-lulc-change-detection/sentinel2_lulc_change_dataset_v2
TOP-LEVEL DATASET BREAKDOWN
2020                          63 files      0.475 GB
2025                          63 files      0.451 GB
labels_2020                   63 files      0.001 GB
metadata                       1 files      0.000 GB
patches                     3024 files      1.123 GB
